# 03 — Geographic EDA

**Owner**: Deekshitha (C5)  
**Phase**: 1 — Data Ingestion & EDA

This notebook explores geographic patterns in the Pittsburgh EMS & Fire dispatch data.

**Topics covered:**
- Top cities by call volume
- Geographic coverage analysis (census block groups)
- City-level service type breakdown
- Missing geographic data analysis
- Scatter map of call density by block group centroids

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 20)
print('✅ Imports ready')

## 1. Load Data

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PARENT_DIR = PROJECT_ROOT.parent

ems_csv = DATA_DIR / 'EMS_Data.csv'
ems_xlsx = PARENT_DIR / 'EMS_Data.xlsx'

if ems_csv.exists():
    df_ems = pd.read_csv(ems_csv)
    df_fire = pd.read_csv(DATA_DIR / 'Fire_Data.csv')
elif ems_xlsx.exists():
    print('Loading from Excel...')
    df_ems = pd.read_excel(ems_xlsx)
    df_fire = pd.read_excel(PARENT_DIR / 'Fire_Data.xlsx')
else:
    raise FileNotFoundError('No data files found.')

df = pd.concat([df_ems, df_fire], ignore_index=True)
print(f'Total records: {len(df):,}')

## 2. Top Cities by Call Volume

In [ ]:
city_counts = df['city_name'].value_counts().head(20)

fig = px.bar(
    x=city_counts.values,
    y=city_counts.index,
    orientation='h',
    title='Top 20 Cities by Call Volume',
    labels={'x': 'Number of Calls', 'y': 'City'},
    color=city_counts.values,
    color_continuous_scale='Turbo',
)
fig.update_layout(template='plotly_dark', width=900, height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. Geographic Coverage — Census Block Groups

In [ ]:
# How many unique census block groups have data?
total_records = len(df)
has_geoid = df['geoid'].notna().sum()
no_geoid = df['geoid'].isna().sum()
unique_geoids = df['geoid'].nunique()

print(f'Records with geoid: {has_geoid:,} ({has_geoid/total_records*100:.1f}%)')
print(f'Records without geoid: {no_geoid:,} ({no_geoid/total_records*100:.1f}%)')
print(f'Unique census block groups: {unique_geoids:,}')

# Coverage pie chart
fig = px.pie(
    values=[has_geoid, no_geoid],
    names=['Has GeoID', 'Missing GeoID'],
    title='Geographic Data Coverage',
    color_discrete_sequence=['#00CC96', '#EF553B'],
    hole=0.4,
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(template='plotly_dark', width=500, height=400)
fig.show()

## 4. City-Level Service Type Breakdown

In [ ]:
top_cities = df['city_name'].value_counts().head(10).index.tolist()
df_top_cities = df[df['city_name'].isin(top_cities)]

city_svc = df_top_cities.groupby(['city_name', 'service']).size().reset_index(name='count')

fig = px.bar(
    city_svc,
    x='city_name',
    y='count',
    color='service',
    barmode='group',
    title='Top 10 Cities — EMS vs Fire Breakdown',
    labels={'city_name': 'City', 'count': 'Call Count', 'service': 'Service'},
    color_discrete_map={'EMS': '#636EFA', 'Fire': '#EF553B'},
)
fig.update_layout(template='plotly_dark', width=900, height=500)
fig.show()

## 5. Missing Geographic Data Analysis

In [ ]:
# Which cities have the most missing geoid data?
geo_missing = df.groupby('city_name').apply(
    lambda g: pd.Series({
        'total': len(g),
        'missing_geoid': g['geoid'].isna().sum(),
        'missing_pct': g['geoid'].isna().mean() * 100,
        'missing_lon': g['census_block_group_center__x'].isna().sum(),
        'missing_lat': g['census_block_group_center__y'].isna().sum(),
    })
).reset_index()

geo_missing = geo_missing.sort_values('missing_geoid', ascending=False).head(15)
geo_missing

In [ ]:
fig = px.bar(
    geo_missing,
    x='city_name',
    y='missing_pct',
    title='Top 15 Cities — Missing GeoID Percentage',
    labels={'city_name': 'City', 'missing_pct': 'Missing GeoID (%)'},
    color='missing_pct',
    color_continuous_scale='Reds',
)
fig.update_layout(template='plotly_dark', width=900, height=450)
fig.show()

## 6. Scatter Map — Call Density by Block Group Centroid

In [ ]:
# Aggregate calls per block group centroid
df_geo = df.dropna(subset=['census_block_group_center__x', 'census_block_group_center__y', 'geoid'])
geo_agg = df_geo.groupby('geoid').agg(
    call_count=('geoid', 'size'),
    lon=('census_block_group_center__x', 'first'),
    lat=('census_block_group_center__y', 'first'),
    top_city=('city_name', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
).reset_index()

print(f'Block groups with data: {len(geo_agg):,}')
print(f'Total calls mapped: {geo_agg["call_count"].sum():,}')

fig = px.scatter_mapbox(
    geo_agg,
    lat='lat',
    lon='lon',
    size='call_count',
    color='call_count',
    hover_name='geoid',
    hover_data=['top_city', 'call_count'],
    title='Call Density by Census Block Group',
    color_continuous_scale='YlOrRd',
    size_max=15,
    zoom=10,
    center={'lat': geo_agg['lat'].mean(), 'lon': geo_agg['lon'].mean()},
    mapbox_style='carto-darkmatter',
)
fig.update_layout(width=900, height=600, template='plotly_dark')
fig.show()

In [ ]:
# Top 10 hotspot block groups
print('=== Top 10 Hotspot Block Groups ===')
top_geoids = geo_agg.nlargest(10, 'call_count')[['geoid', 'top_city', 'call_count', 'lat', 'lon']]
top_geoids

In [ ]:
print('=== Geographic Summary ===')
print(f'Total unique cities: {df["city_name"].nunique()}')
print(f'Total unique city codes: {df["city_code"].nunique()}')
print(f'Total unique census block groups: {df["geoid"].nunique()}')
print(f'Records with coordinates: {df_geo.shape[0]:,} / {df.shape[0]:,} ({df_geo.shape[0]/df.shape[0]*100:.1f}%)')
print(f'Latitude range: {df_geo["census_block_group_center__y"].min():.4f} — {df_geo["census_block_group_center__y"].max():.4f}')
print(f'Longitude range: {df_geo["census_block_group_center__x"].min():.4f} — {df_geo["census_block_group_center__x"].max():.4f}')